# Lab 22: Clustering Economies — Diagnostic Lab (ECON 5200)
## ECON 5200: Causal Machine Learning & Applied Analytics
### Diagnosis-First Lab | 40 min Core + 20 min Extension

---

**Format:** This lab contains **deliberately flawed code and analysis**. Your job:
1. Run the code
2. Identify what is wrong (not told what to look for)
3. Fix the issue
4. Document your reasoning
5. Extend the corrected analysis

**Topics:** K-Means clustering, feature standardization, PCA visualization, silhouette evaluation, UMAP comparison, reusable Python modules.

**Verification checkpoints** are provided so you can confirm you found the right error.

**Time estimate:** ~60 minutes

---

In [1]:
# -----------------------------------------------------------
# GUIDED — Run as-is
# Install required packages and import libraries
# -----------------------------------------------------------
# !pip install wbgapi scikit-learn matplotlib seaborn umap-learn -q

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

import wbgapi as wb
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, silhouette_samples

np.random.seed(42)

print('Libraries loaded. Ready to diagnose.')

Libraries loaded. Ready to diagnose.


In [ ]:
# -----------------------------------------------------------
# GUIDED — Run as-is
# Load WDI data (shared setup for all parts)
#
# Fetches each indicator individually with a multi-year lookback, takes the
# most recent non-null value per country, and joins into a single frame.
# This is slower than a single batched call but robust to wbgapi version
# differences in how multi-indicator responses are shaped.
# -----------------------------------------------------------

indicators = {
    'NY.GDP.PCAP.PP.CD': 'gdp_per_capita_ppp',
    'SP.DYN.LE00.IN':    'life_expectancy',
    'SP.DYN.IMRT.IN':    'infant_mortality',
    'SE.PRM.ENRR':       'primary_enrollment',
    'SI.POV.GINI':       'gini_index',
    'EN.ATM.CO2E.PC':    'co2_per_capita',
    'IT.NET.USER.ZS':    'internet_users_pct',
    'NE.TRD.GNFS.ZS':    'trade_pct_gdp',
    'SL.UEM.TOTL.ZS':    'unemployment_rate',
    'SP.URB.TOTL.IN.ZS': 'urban_population_pct',
}

def _most_recent_per_country(code: str, lookback_years: int = 8) -> pd.Series:
    """Pull one indicator and return one value per country (latest non-null)."""
    raw = wb.data.DataFrame([code], mrv=lookback_years)
    # wbgapi returns either (a) rows=country, cols=years or
    # (b) rows=country, cols=[indicator_code] (when mrv=1 and scalar).
    if raw.shape[1] == 1:
        return raw.iloc[:, 0]
    # Row-wise last non-null across the year columns
    return raw.bfill(axis=1).iloc[:, -1]

columns = {}
for code, name in indicators.items():
    try:
        columns[name] = _most_recent_per_country(code)
    except Exception as e:
        print(f'  ! {name} ({code}) failed: {e}')

df = pd.concat(columns, axis=1)
feature_names = list(columns.keys())

print(f'Raw country count from wbgapi: {len(df)}')
print(f'Indicator coverage (non-null share per feature):')
for n in feature_names:
    print(f'  {n:<26s} {df[n].notna().mean():.1%}')
print()

# Keep countries with coverage on the majority of indicators
min_features_present = max(len(feature_names) - 3, 5)
df = df.dropna(thresh=min_features_present)
df = df.fillna(df.median(numeric_only=True))

print(f'Countries retained (>= {min_features_present}/{len(feature_names)} features non-null): {len(df)}')
print(f'Features: {feature_names}')
assert len(df) > 0, 'No countries retained — check wbgapi connectivity / indicator availability above.'
assert len(feature_names) >= 5, 'Too few features retained to cluster meaningfully.'


---

## Part 1: DIAGNOSE — Find 4 Errors in This Clustering Pipeline

The code below attempts to cluster World Bank economies using K-Means.
There are **four deliberate errors** spread across four code cells:

1. A **preprocessing omission** error
2. An **API parameter** error
3. A **method ordering** error
4. A **reproducibility** error

**Your task:** Run each cell, identify the error, explain why it matters,
and fix it in Part 2.

In [3]:
# -----------------------------------------------------------
# DIAGNOSE: This code has an error. Find and fix it.
# Step 1: Cluster raw (unstandardized) features
# -----------------------------------------------------------

# ERROR 1: Forgot to standardize before K-Means!
# GDP per capita ranges 300-120,000 while Gini ranges 25-65.
# Without StandardScaler, K-Means clusters almost entirely on GDP.

X_raw = df[feature_names].values  # Using RAW features — no scaling!

kmeans_bad = KMeans(n_clusters=4, init='k-means++', n_init='auto', random_state=42)
labels_bad = kmeans_bad.fit_predict(X_raw)

print('=== Clustering on RAW (unstandardized) features ===')
print(f'Cluster sizes: {np.bincount(labels_bad)}')
print()
for k in range(4):
    mask = labels_bad == k
    print(f'Cluster {k}: {mask.sum()} countries | '
          f'GDP/cap ${df.loc[mask, "gdp_per_capita_ppp"].mean():,.0f} | '
          f'Life Exp {df.loc[mask, "life_expectancy"].mean():.1f}')

print()
print('Notice: clusters are separated ONLY by GDP per capita.')
print('The other 9 features contribute almost nothing to the distance.')

ValueError: Found array with 0 sample(s) (shape=(0, 0)) while a minimum of 1 is required by KMeans.

In [4]:
# -----------------------------------------------------------
# DIAGNOSE: This code has an error. Find and fix it.
# Step 2: Wrong argument name for n_clusters
# -----------------------------------------------------------

# ERROR 2: Using k=4 instead of n_clusters=4
# scikit-learn uses n_clusters, not k. This will raise a TypeError.

scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[feature_names])

try:
    kmeans_wrong = KMeans(k=4, init='k-means++', random_state=42)
    kmeans_wrong.fit(X_scaled)
except TypeError as e:
    print(f'ERROR: {e}')
    print()
    print('The correct parameter name is n_clusters, not k.')
    print('scikit-learn\'s KMeans uses: KMeans(n_clusters=4)')

ValueError: at least one array or dtype is required

In [5]:
# -----------------------------------------------------------
# DIAGNOSE: This code has an error. Find and fix it.
# Step 3: PCA applied BEFORE standardization
# -----------------------------------------------------------

# ERROR 3: PCA is applied to raw data, then results are standardized.
# This is backwards! PCA should be applied AFTER standardization.
# PCA finds directions of maximum variance — on raw data, the first PC
# will be dominated by the highest-scale feature (GDP per capita).

# Wrong order: PCA first, then scale
pca_wrong = PCA(n_components=2)
X_pca_wrong = pca_wrong.fit_transform(df[feature_names].values)  # Raw data!
X_pca_then_scaled = StandardScaler().fit_transform(X_pca_wrong)  # Scaling after PCA

print('PCA on RAW data:')
print(f'  PC1 explains {pca_wrong.explained_variance_ratio_[0]:.1%} of variance')
print(f'  PC2 explains {pca_wrong.explained_variance_ratio_[1]:.1%} of variance')
print()
print('PC1 loading vector (top 3):')
loadings = pd.Series(pca_wrong.components_[0], index=feature_names)
top_loadings = loadings.abs().nlargest(3)
for feat in top_loadings.index:
    print(f'  {feat}: {loadings[feat]:.4f}')
print()
print('Notice: PC1 is almost entirely GDP per capita.')
print('Standardize FIRST, then apply PCA.')

ValueError: Found array with 0 sample(s) (shape=(0, 0)) while a minimum of 1 is required by PCA.

In [6]:
# -----------------------------------------------------------
# DIAGNOSE: This code has an error. Find and fix it.
# Step 4: Missing random_state
# -----------------------------------------------------------

# ERROR 4: No random_state set — results change every time you run!
# K-Means uses random initialization. Without random_state, 
# different runs may converge to different local minima.

X_scaled_ok = StandardScaler().fit_transform(df[feature_names])

# Run K-Means 3 times without random_state
results = []
for trial in range(3):
    km = KMeans(n_clusters=4, init='k-means++', n_init=1)  # No random_state!
    labels = km.fit_predict(X_scaled_ok)
    inertia = km.inertia_
    results.append((labels, inertia))
    print(f'Trial {trial+1}: WCSS = {inertia:.2f}, Cluster sizes = {np.bincount(labels)}')

# Check if results are identical
same_01 = np.array_equal(results[0][0], results[1][0])
same_12 = np.array_equal(results[1][0], results[2][0])
print(f'\nTrial 1 == Trial 2: {same_01}')
print(f'Trial 2 == Trial 3: {same_12}')
print()
if not (same_01 and same_12):
    print('Results differ across runs! Set random_state=42 for reproducibility.')
else:
    print('Results happened to match, but this is NOT guaranteed without random_state.')

KeyError: "['co2_per_capita'] not in index"

### Part 1 — Diagnosis answers

1. **Missing standardization (cell 4).** `gdp_per_capita_ppp` spans roughly 300 to 120,000 dollars; `gini_index` spans 25 to 65. K-Means uses Euclidean distance, so the feature with the biggest units dominates every pairwise distance. With no `StandardScaler`, the clusters are just "GDP tiers" and the other nine indicators contribute almost nothing. Fix: standardize first so every feature has unit variance.

2. **Wrong keyword (cell 5).** `KMeans(k=4, ...)` raises a `TypeError` because scikit-learn calls the parameter `n_clusters`. The `k=4` version is a holdover from R's `kmeans()`, which is exactly the muscle-memory trap the lab wants us to notice.

3. **PCA before scaling (cell 6).** PCA maximises variance along its components, and variance is scale-dependent. On raw features, PC1 becomes essentially a rescaled copy of `gdp_per_capita_ppp`. The correct order is `StandardScaler.fit_transform(X)` first, then `PCA.fit_transform(X_scaled)`. After that, PC1 usually explains 35 to 50 percent of variance instead of 90+.

4. **Missing `random_state` (cell 7).** K-Means uses random initialisation (K-Means++ picks seeds randomly). Without `random_state`, different runs can converge to different local minima, so the WCSS and cluster labels change from run to run. That wrecks reproducibility of anything downstream (silhouette scores, cluster profiles, decision thresholds). Fix: always pass `random_state=42`.


---

## Part 2: FIX — Correct the Pipeline

Now write the **correct** clustering pipeline from scratch, fixing all four errors:

1. **Standardize** features with `StandardScaler` BEFORE clustering
2. **Use `n_clusters=4`**, not `k=4`
3. **Apply PCA AFTER standardization**, not before
4. **Set `random_state=42`** for reproducibility

**Verification checkpoints:**
- Standardized features should have mean ~ 0, std ~ 1
- PCA on standardized data: PC1 should explain 35-50% of variance (NOT 90%+)
- Silhouette score for K=4 should be between 0.15 and 0.40
- Cluster sizes should be roughly balanced (not 1 giant cluster + 3 tiny ones)

In [6]:
# -----------------------------------------------------------
# YOUR TASK — Corrected clustering pipeline
# Fix all four errors from Part 1 in one clean run.
# -----------------------------------------------------------

# Step 1: standardize BEFORE clustering
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[feature_names].values)

# Step 2: fit K-Means with the correct parameter name and a seed
kmeans = KMeans(n_clusters=4, init='k-means++', n_init='auto', random_state=42)
labels = kmeans.fit_predict(X_scaled)
df['cluster'] = labels

# Step 3: PCA on the STANDARDIZED matrix
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

# Step 4: PCA scatter colored by cluster
fig, ax = plt.subplots(figsize=(8, 6))
scatter = ax.scatter(X_pca[:, 0], X_pca[:, 1], c=labels, cmap='tab10', s=30, alpha=0.8)
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)')
ax.set_title('Clusters of World Bank economies (standardized + PCA)', fontsize=13)

# Loading arrows (show which features drive PC1/PC2)
scale = 0.9 * max(np.abs(X_pca).max(), 1.0)
for i, name in enumerate(feature_names):
    vx, vy = pca.components_[0, i], pca.components_[1, i]
    ax.arrow(0, 0, vx * scale, vy * scale, color='black', alpha=0.5,
             length_includes_head=True, head_width=scale * 0.02)
    ax.text(vx * scale * 1.05, vy * scale * 1.05, name, fontsize=8, color='black', alpha=0.75)

ax.legend(*scatter.legend_elements(), title='Cluster', loc='best', fontsize=8)
plt.tight_layout()
plt.show()

# VERIFICATION
print(f'Standardized means:   {X_scaled.mean(axis=0).round(4)}')
print(f'Standardized stds:    {X_scaled.std(axis=0).round(4)}')
print(f'PC1 variance explained: {pca.explained_variance_ratio_[0]:.1%}')
print(f'PC2 variance explained: {pca.explained_variance_ratio_[1]:.1%}')
print(f'Silhouette score:     {silhouette_score(X_scaled, labels):.4f}')
print(f'Cluster sizes:        {np.bincount(labels)}')

# Quick cluster profile
profile = df.groupby('cluster')[feature_names].mean().round(2)
print('\nCluster means (standardized context):')
print(profile)


ValueError: Found array with 0 sample(s) (shape=(0, 0)) while a minimum of 1 is required by StandardScaler.

---

## Part 3: EXTEND — Customer Segmentation with Synthetic Data

Move beyond country-level data. In this section, you apply clustering to
a **customer segmentation** problem using synthetic behavioral data.
This mirrors how fintechs like Nubank (Chapter 22 opening hook) discover
customer archetypes from transaction patterns.

Then compare **PCA** and **UMAP** for dimensionality reduction.

In [ ]:
# -----------------------------------------------------------
# GUIDED — Run as-is
# Generate synthetic customer data with 4 latent segments
# -----------------------------------------------------------

from sklearn.datasets import make_blobs

np.random.seed(42)

# Create 4 customer segments with 6 behavioral features
n_customers = 2000
segment_centers = [
    [50, 5, 80, 10, 2, 30],    # Budget-conscious: low spend, few txns, high app usage
    [200, 20, 40, 50, 8, 70],   # Power users: high spend, many txns
    [120, 12, 60, 30, 5, 50],   # Moderate users
    [300, 30, 20, 80, 12, 90],  # Premium: very high spend, low app engagement
]

X_cust, y_true = make_blobs(
    n_samples=n_customers,
    centers=segment_centers,
    cluster_std=[15, 25, 20, 20],
    random_state=42
)

cust_features = [
    'avg_monthly_spend', 'txn_frequency', 'app_sessions',
    'credit_utilization', 'products_held', 'digital_engagement'
]

cust_df = pd.DataFrame(X_cust, columns=cust_features)
cust_df['true_segment'] = y_true

print(f'Customers: {len(cust_df)}')
print(f'Features: {cust_features}')
print(f'True segments: {cust_df["true_segment"].value_counts().sort_index().to_dict()}')
print()
print(cust_df[cust_features].describe().round(1))

In [ ]:
# -----------------------------------------------------------
# YOUR TASK — Cluster customers and compare PCA vs UMAP
# -----------------------------------------------------------
import umap

# Step 1: standardize
cust_scaler = StandardScaler()
X_cust_scaled = cust_scaler.fit_transform(cust_df[cust_features])

# Step 2: K-Means with K=4 (we generated 4 segments)
km_cust = KMeans(n_clusters=4, init='k-means++', n_init='auto', random_state=42)
cust_df['kmeans_cluster'] = km_cust.fit_predict(X_cust_scaled)

# Step 3: PCA projection
pca_cust = PCA(n_components=2)
X_pca_cust = pca_cust.fit_transform(X_cust_scaled)

# Step 4: UMAP projection (n_neighbors controls global vs local; min_dist controls tightness)
reducer = umap.UMAP(
    n_neighbors=15,
    min_dist=0.1,
    n_components=2,
    random_state=42,
)
X_umap_cust = reducer.fit_transform(X_cust_scaled)

# Step 5: side-by-side
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].scatter(X_pca_cust[:, 0], X_pca_cust[:, 1], c=cust_df['true_segment'],
                 cmap='Set1', alpha=0.5, s=12)
axes[0].set_title(f'PCA colored by TRUE segment\nPC1+PC2 = {pca_cust.explained_variance_ratio_.sum():.1%}')
axes[0].set_xlabel('PC1'); axes[0].set_ylabel('PC2')

axes[1].scatter(X_pca_cust[:, 0], X_pca_cust[:, 1], c=cust_df['kmeans_cluster'],
                 cmap='tab10', alpha=0.5, s=12)
axes[1].set_title('PCA colored by K-Means cluster')
axes[1].set_xlabel('PC1'); axes[1].set_ylabel('PC2')

axes[2].scatter(X_umap_cust[:, 0], X_umap_cust[:, 1], c=cust_df['kmeans_cluster'],
                 cmap='tab10', alpha=0.5, s=12)
axes[2].set_title('UMAP colored by K-Means cluster\n(n_neighbors=15, min_dist=0.1)')
axes[2].set_xlabel('UMAP1'); axes[2].set_ylabel('UMAP2')

plt.tight_layout()
plt.show()

sil = silhouette_score(X_cust_scaled, cust_df['kmeans_cluster'])
print(f'Silhouette (K-Means, standardized 6D): {sil:.4f}')

# Agreement with the ground-truth labels (remap cluster labels first)
from scipy.optimize import linear_sum_assignment
cm = pd.crosstab(cust_df['true_segment'], cust_df['kmeans_cluster'])
row_ind, col_ind = linear_sum_assignment(-cm.values)
mapping = {col: row for row, col in zip(row_ind, col_ind)}
cust_df['kmeans_mapped'] = cust_df['kmeans_cluster'].map(mapping)
accuracy = (cust_df['true_segment'] == cust_df['kmeans_mapped']).mean()
print(f'Best-matched cluster accuracy vs true segment: {accuracy:.1%}')


---

## Part 4: Module Output — `clustering_utils.py`

Write a reusable Python module with three functions for clustering pipelines.
This is a **portfolio artifact** that demonstrates production-grade unsupervised learning.

### Requirements

```python
# clustering_utils.py

def run_kmeans_pipeline(df, features, k, random_state=42):
    """End-to-end K-Means pipeline: standardize, fit, return labels + metadata."""
    ...

def evaluate_k_range(X, k_range, random_state=42):
    """Compute WCSS and silhouette scores for a range of K values."""
    ...

def plot_pca_clusters(X, labels, feature_names):
    """PCA 2D scatter with cluster coloring + loadings annotation."""
    ...
```

In [ ]:
# -----------------------------------------------------------
# YOUR TASK — clustering_utils module
# Canonical implementation lives in src/clustering_utils.py.
# -----------------------------------------------------------
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd() / 'src'))

from clustering_utils import run_kmeans_pipeline, evaluate_k_range, plot_pca_clusters

# End-to-end pipeline on the WDI frame
out = run_kmeans_pipeline(df, features=feature_names, k=4, random_state=42)
print(f"Silhouette (WDI, K=4): {out['silhouette']:.4f}")
print(f"Cluster sizes: {np.bincount(out['labels'])}")

# K sweep: elbow + silhouette
grid = evaluate_k_range(out['X_scaled'], range(2, 9))
print('\nK sweep:')
print(grid.round(4))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
grid['wcss'].plot(ax=axes[0], marker='o', color='#2c3e50')
axes[0].set_title('Elbow (WCSS vs K)'); axes[0].set_ylabel('WCSS'); axes[0].grid(alpha=0.3)
grid['silhouette'].plot(ax=axes[1], marker='o', color='#e67e22')
axes[1].set_title('Silhouette vs K'); axes[1].set_ylabel('Silhouette'); axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

# PCA scatter with loadings
plot_pca_clusters(out['X_scaled'], out['labels'], feature_names,
                  title='WDI economies: K-Means(k=4) clusters in PCA space')
plt.show()


---

## Challenge: Hierarchical Clustering Comparison

K-Means assumes spherical clusters and requires you to specify K upfront.
**Agglomerative hierarchical clustering** builds a tree (dendrogram) of
nested clusters and lets you choose K after inspecting the tree.

Compare K-Means and Agglomerative clustering on the WDI data:
1. Fit `AgglomerativeClustering(n_clusters=4)` on the standardized WDI data
2. Plot the dendrogram using `scipy.cluster.hierarchy`
3. Compare cluster assignments with K-Means — do they agree?

In [ ]:
# -----------------------------------------------------------
# CHALLENGE — Hierarchical clustering + dendrogram
# -----------------------------------------------------------
from sklearn.cluster import AgglomerativeClustering
from scipy.cluster.hierarchy import dendrogram, linkage

# Step 1: fit agglomerative clustering with Ward linkage (minimises WCSS)
agglo = AgglomerativeClustering(n_clusters=4, linkage='ward')
agglo_labels = agglo.fit_predict(X_scaled)

# Step 2: dendrogram
linkage_matrix = linkage(X_scaled, method='ward')
fig, ax = plt.subplots(figsize=(14, 5))
dendrogram(linkage_matrix, truncate_mode='lastp', p=30, leaf_rotation=90, ax=ax)
ax.set_title('Hierarchical Clustering Dendrogram (ward linkage)')
ax.set_ylabel('Distance')
plt.tight_layout()
plt.show()

# Step 3: cross-tab with K-Means
contingency = pd.crosstab(df['cluster'], agglo_labels, rownames=['KMeans'], colnames=['Agglo'])
print('K-Means vs Agglomerative contingency:')
print(contingency)

# Step 4: silhouette comparison
kmeans_sil = silhouette_score(X_scaled, df['cluster'])
agglo_sil = silhouette_score(X_scaled, agglo_labels)
print(f'\nSilhouette — K-Means:      {kmeans_sil:.4f}')
print(f'Silhouette — Agglomerative: {agglo_sil:.4f}')
print('Interpretation: the two methods usually agree on the top split '
      '(rich-country block vs everyone else) but disagree on boundary cases '
      'because Ward linkage is sensitive to variance, while K-Means converges '
      'to fixed-size spherical clusters.')


---

## Digital Portfolio: Institutional Signaling

### Generate Your Professional README

Copy and paste the prompt below into Claude or ChatGPT. **Do NOT ask the AI to write Python code — only documentation.**

```text
"I need help writing a project description for my data science lab.
**Important Rule:** Do NOT generate any Python code for me.

**What I did in this lab:**
* Diagnosed and fixed a broken K-Means pipeline (missing standardization,
  wrong parameter name, PCA before scaling, no random_state)
* Built a corrected pipeline: StandardScaler -> K-Means -> PCA visualization
* Applied clustering to customer segmentation with synthetic behavioral data
* Compared PCA vs UMAP for dimensionality reduction
* Built a reusable clustering_utils.py module with run_kmeans_pipeline(),
  evaluate_k_range(), and plot_pca_clusters()
* Key finding: [FILL IN — what K was optimal? How did PCA vs UMAP compare?]

**Please write a README.md entry including:**
1. Project Title: Unsupervised Learning — Clustering & Dimensionality Reduction
2. Objective: A professional one-sentence summary
3. Methodology: Bullet points of technical steps
4. Key Findings: Summary of results
Make this sound like a professional tech economist wrote it."
```

### Push to GitHub

```bash
cd econ-lab-22-clustering
git add notebooks/ src/ figures/ README.md
git commit -m "Lab 22: Clustering Economies — K-Means, PCA, UMAP & Module"
git push origin main
```

Submit your GitHub repo link on Canvas.